## Pt2 

SIMILITUD DE USUARIOS

1. Càrrega i preparació de dades: S'importen dades de valoracions de pel·lícules.
2. Càlcul de similitud entre usuaris:
Es defineixen funcions per calcular la similitud entre usuaris (ex. usant una matriu de similitud).
Es compara l'eficiència de diferents mètodes per calcular aquesta similitud.
3. Recomanació de pel·lícules:
Es crea una funció que identifica els usuaris més semblants a un usuari donat.
Es desenvolupa un mètode per predir la puntuació d'una pel·lícula per a un usuari basant-se en les puntuacions dels usuaris similars (mitjana ponderada).
Es crea una funció que genera una llista de recomanacions per a un usuari.
4. Avaluació del sistema:
Es divideix el conjunt de dades en conjunts de train i test.
Es defineix una mètrica per avaluar el sistema (MAE).
S'avalua el sistema utilitzant la MAE en el conjunt de test.
5. Anàlisi addicional:
S'analitza si és més efectiu crear un recomanador únic o separats per a diferents grups (homes i dones en aquest cas).
Objectius de la pràctica:
Entendre els principis bàsics dels sistemes de recomanació basats en usuaris.
Aprendre a calcular la similitud entre usuaris.
Desenvolupar un sistema de recomanació funcional.
Avaluar el rendiment del sistema.
Analitzar la influència de factors demogràfics en el sistema de recomanació.

### 1. Càrrega i preparació de dades: S'importen dades de valoracions de pel·lícules.

In [4]:
# Cel·la 1: Importacions i constants
import math
import time 


RATINGS_FILE = 'ml-1m/ratings.dat'
MOVIES_FILE = 'ml-1m/movies.dat'
USERS_FILE = 'ml-1m/users.dat'

### 2. Càlcul de similitud entre usuaris:
Es defineixen funcions per calcular la similitud entre usuaris (ex. usant una matriu de similitud).
Es compara l'eficiència de diferents mètodes per calcular aquesta similitud.

In [6]:
# Cel·la 2: Càrrega i preparació de dades
# (Implementació corresponent a la Secció 3.3 'Recolección de Datos'
# i 2.6 'Metodología Clave: Validación' del resum)
import random

def load_ratings_data(file_path, train_test_split=0.8):
    """
    Càrrega les dades de ratings des del fitxer .dat.
    Separa les dades en conjunts d'entrenament i prova.
    Retorna dos diccionaris: users_ratings_train i users_ratings_test.
    Format: {userID: {movieID: rating}}
    """
    users_ratings_train = {}
    users_ratings_test = {}
    
    print(f"Carregant dades des de {file_path}...")
    
    try:
        with open(file_path, 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                
                # Format: UserID::MovieID::Rating::Timestamp
                parts = line.split('::')
                try:
                    user_id = int(parts[0])
                    movie_id = int(parts[1])
                    rating = float(parts[2])
                except ValueError:
                    print(f"Ometent línia mal formada: {line.strip()}")
                    continue

                # Decidim si va a train o a test
                if random.random() < train_test_split:
                    # Afegir a train
                    if user_id not in users_ratings_train:
                        users_ratings_train[user_id] = {}
                    users_ratings_train[user_id][movie_id] = rating
                else:
                    # Afegir a test
                    if user_id not in users_ratings_test:
                        users_ratings_test[user_id] = {}
                    users_ratings_test[user_id][movie_id] = rating

        print("Càrrega de dades completada.")
        print(f"Usuaris a train: {len(users_ratings_train)}")
        print(f"Usuaris a test: {len(users_ratings_test)}")
        
        # Assegurem que tots els usuaris de test existeixen a train 
        # (per poder fer recomanacions)
        # Els usuaris que només tenen ratings a test es mouen a train.
        test_users_only = set(users_ratings_test.keys()) - set(users_ratings_train.keys())
        for user_id in test_users_only:
            users_ratings_train[user_id] = users_ratings_test.pop(user_id)
            
        print(f"Usuaris moguts de test a train (per no estar a train): {len(test_users_only)}")
        print(f"Usuaris finals a train: {len(users_ratings_train)}")
        print(f"Usuaris finals a test: {len(users_ratings_test)}")

        return users_ratings_train, users_ratings_test

    except FileNotFoundError:
        print(f"ERROR: Fitxer no trobat a la ruta: {file_path}")
        return None, None
    except Exception as e:
        print(f"S'ha produït un error inesperat durant la càrrega: {e}")
        return None, None

# Executem la càrrega de dades
users_ratings_train, users_ratings_test = load_ratings_data(RATINGS_FILE)

Carregant dades des de ml-1m/ratings.dat...
Càrrega de dades completada.
Usuaris a train: 6040
Usuaris a test: 6038
Usuaris moguts de test a train (per no estar a train): 0
Usuaris finals a train: 6040
Usuaris finals a test: 6038


In [11]:
# Definicions de les funcions de similitud

def similarity_euclidean(user1_ratings, user2_ratings):
    """
    Calcula la similitud basada en la distància euclidiana entre dos usuaris.
    Entrada: Diccionaris {movieID: rating} per a cada usuari.
    """
    # Trobar ítems valorats en comú
    common_items = set(user1_ratings.keys()) & set(user2_ratings.keys())
    
    # Si no hi ha ítems en comú, no hi ha similitud
    if not common_items:
        return 0.0
        
    # Calcular la suma de les diferències al quadrat
    sum_of_squares = sum(
        (user1_ratings[item] - user2_ratings[item])**2 
        for item in common_items
    )
    
    # Calcular la distància euclidiana
    distance = math.sqrt(sum_of_squares)
    
    # Convertir la distància (mètrica de dissimilaritat) 
    # en una puntuació de similitud (on 1 és idèntic i 0 és distant)
    return 1.0 / (1.0 + distance)

def similarity_pearson(user1_ratings, user2_ratings):
    """
    Calcula el coeficient de correlació de Pearson entre dos usuaris.
    Entrada: Diccionaris {movieID: rating} per a cada usuari.
    """
    # Trobar ítems valorats en comú
    common_items = set(user1_ratings.keys()) & set(user2_ratings.keys())
    n = len(common_items)
    
    # Es necessiten almenys 2 punts en comú per calcular la correlació
    if n < 2:
        return 0.0
        
    # Calcular les mitjanes dels ratings *només* per als ítems en comú
    mean1 = sum(user1_ratings[item] for item in common_items) / n
    mean2 = sum(user2_ratings[item] for item in common_items) / n
    
    # Calcular numerador (covariància) i 
    # denominador (producte de desviacions estàndard)
    numerator = 0.0
    sum_sq1 = 0.0
    sum_sq2 = 0.0
    
    for item in common_items:
        dev1 = user1_ratings[item] - mean1
        dev2 = user2_ratings[item] - mean2
        numerator += dev1 * dev2
        sum_sq1 += dev1**2
        sum_sq2 += dev2**2
        
    denominator = math.sqrt(sum_sq1 * sum_sq2)
    
    # Evitar divisió per zero (si no hi ha variància en els ratings comuns)
    if denominator == 0:
        return 0.0 
        
    return numerator / denominator

print("Funcions de similitud (similarity_euclidean, similarity_pearson) definides.")

Funcions de similitud (similarity_euclidean, similarity_pearson) definides.


#### Comparamos la eficiencia

In [12]:
# Cel·la 4: Càlcul de la matriu de similitud i comparació d'eficiència

def build_similarity_matrix(users_ratings, similarity_function, max_users=None):
    """
    Construeix una matriu de similitud (com a dict de dicts)
    per als usuaris donats, utilitzant la funció de similitud especificada.
    Retorna la matriu i el temps emprat.
    """
    print(f"Iniciant càlcul de la matriu amb: {similarity_function.__name__}...")
    start_time = time.time()
    
    similarity_matrix = {}
    
    if max_users:
        user_ids = sorted(list(users_ratings.keys()))[:max_users]
    else:
        user_ids = sorted(list(users_ratings.keys()))
    
    num_users = len(user_ids)
    
    for i in range(num_users):
        user_i_id = user_ids[i]
        similarity_matrix[user_i_id] = {}
        user_i_ratings = users_ratings[user_i_id]
        
        for j in range(i, num_users):
            user_j_id = user_ids[j]
            
            if i == j:
                sim = 1.0
            else:
                user_j_ratings = users_ratings[user_j_id]
                sim = similarity_function(user_i_ratings, user_j_ratings)
            
            similarity_matrix[user_i_id][user_j_id] = sim
            
            if user_j_id not in similarity_matrix:
                similarity_matrix[user_j_id] = {}
            similarity_matrix[user_j_id][user_i_id] = sim

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Càlcul finalitzat. Temps emprat: {elapsed_time:.4f} segons.")
    
    return similarity_matrix, elapsed_time

# --- Comparació d'Eficiència ---
# Aquesta cel·la ara pot executar-se correctament, ja que 'users_ratings_train'
# s'ha definit a la Cel·la 2.

N_USERS_COMPARISON = 500 

if 'users_ratings_train' in locals() and users_ratings_train is not None:
    print(f"--- Iniciant Comparació d'Eficiència (Mostra de {N_USERS_COMPARISON} usuaris) ---")
    
    # Mètode 1: Similitud Euclidiana
    sim_matrix_euclidean, time_euclidean = build_similarity_matrix(
        users_ratings_train, 
        similarity_euclidean, 
        max_users=N_USERS_COMPARISON
    )
    
    # Mètode 2: Correlació de Pearson
    sim_matrix_pearson, time_pearson = build_similarity_matrix(
        users_ratings_train, 
        similarity_pearson, 
        max_users=N_USERS_COMPARISON
    )
    
    print("\n--- Resultats de la Comparació d'Eficiència ---")
    print(f"Mida de la prova: {N_USERS_COMPARISON} usuaris")
    print(f"Temps (Similitud Euclidiana): \t{time_euclidean:.4f} segons")
    print(f"Temps (Correlació Pearson): \t{time_pearson:.4f} segons")

else:
    print("ERROR: La variable 'users_ratings_train' no s'ha pogut inicialitzar.")
    print("Si us plau, revisi la sortida de la 'Cel·la 2: Càrrega i preparació de dades'.")

--- Iniciant Comparació d'Eficiència (Mostra de 500 usuaris) ---
Iniciant càlcul de la matriu amb: similarity_euclidean...
Càlcul finalitzat. Temps emprat: 1.6464 segons.
Iniciant càlcul de la matriu amb: similarity_pearson...
Càlcul finalitzat. Temps emprat: 2.2488 segons.

--- Resultats de la Comparació d'Eficiència ---
Mida de la prova: 500 usuaris
Temps (Similitud Euclidiana): 	1.6464 segons
Temps (Correlació Pearson): 	2.2488 segons


### 3. Recomanació de pel·lícules:
Es crea una funció que identifica els usuaris més semblants a un usuari donat.
Es desenvolupa un mètode per predir la puntuació d'una pel·lícula per a un usuari basant-se en les puntuacions dels usuaris similars (mitjana ponderada).
Es crea una funció que genera una llista de recomanacions per a un usuari.

In [24]:
# Cel·la 5: Funcions de Recomanació (CORREGIDA)

def get_k_nearest_neighbors(target_user_id, similarity_matrix, k=25):
    """
    Identifica els k usuaris més semblants (veïns) a un usuari objectiu.
    """
    if target_user_id not in similarity_matrix:
        print(f"Advertència: L'usuari {target_user_id} no es troba a la matriu de similitud.")
        return []
        
    sims = similarity_matrix[target_user_id]
    neighbors = [(sim, user_id) for user_id, sim in sims.items() if user_id != target_user_id]
    neighbors.sort(reverse=True)
    return neighbors[:k]

def predict_rating(target_user_id, movie_id, users_ratings, k_nearest_neighbors):
    """
    Prediu la puntuació d'una pel·lícula per a un usuari, basant-se 
    en la mitjana ponderada dels seus veïns.
    (Implementació Secció 3.2.1)
    """
    weighted_sum = 0.0
    similarity_sum = 0.0
    
    for similarity, neighbor_id in k_nearest_neighbors:
        
        # --- CORRECCIÓ ---
        # Ignorem els veïns que no són realment similars 
        # (correlació negativa o zero).
        if similarity <= 0:
            continue
        # --- FI DE LA CORRECCIÓ ---
            
        # Comprovem si el veí ha valorat la pel·lícula
        if neighbor_id in users_ratings and movie_id in users_ratings[neighbor_id]:
            neighbor_rating = users_ratings[neighbor_id][movie_id]
            
            # Ponderem la puntuació del veí per la seva similitud
            weighted_sum += similarity * neighbor_rating
            similarity_sum += similarity
            
    # Evitem la divisió per zero
    if similarity_sum == 0:
        return 0.0
        
    # Retornem la mitjana ponderada
    return weighted_sum / similarity_sum

def get_recommendations(target_user_id, similarity_matrix, users_ratings_train, k=25, n_recommendations=10):
    """
    Genera una llista de N recomanacions de pel·lícules per a un usuari.
    """
    
    # 1. Trobar els veïns més propers
    try:
        k_neighbors = get_k_nearest_neighbors(target_user_id, similarity_matrix, k)
        if not k_neighbors:
            print(f"No s'han trobat veïns per a l'usuari {target_user_id}. No es poden generar recomanacions.")
            return []
    except Exception as e:
        print(f"Error en obtenir veïns: {e}")
        return []

    # 2. Identificar pel·lícules candidates
    user_seen_movies = set()
    if target_user_id in users_ratings_train:
        user_seen_movies = set(users_ratings_train[target_user_id].keys())
        
    candidate_movies = set()
    for _, neighbor_id in k_neighbors:
        # CORRECCIÓ: No té sentit mirar les pel·lícules 
        # de veïns amb similitud negativa
        
        # Comprovem la similitud directament a la matriu
        if target_user_id in similarity_matrix and neighbor_id in similarity_matrix[target_user_id]:
            sim = similarity_matrix[target_user_id][neighbor_id]
            
            if sim > 0 and neighbor_id in users_ratings_train:
                candidate_movies.update(users_ratings_train[neighbor_id].keys())
        
    candidate_movies = candidate_movies - user_seen_movies
    
    if not candidate_movies:
        print(f"L'usuari {target_user_id} ja ha vist totes les pel·lícules vistes pels seus veïns (positius).")
        return []

    # 3. Predir la puntuació per a cada pel·lícula candidata
    predictions = []
    for movie_id in candidate_movies:
        predicted = predict_rating(target_user_id, movie_id, users_ratings_train, k_neighbors)
        
        if predicted > 0:
            predictions.append((predicted, movie_id))
            
    # 4. Ordenar les prediccions (descendent) i retornar les N millors
    predictions.sort(reverse=True)
    
    return predictions[:n_recommendations]

print("Funcions (CORREGIDES) get_k_nearest_neighbors, predict_rating, i get_recommendations definides.")

Funcions (CORREGIDES) get_k_nearest_neighbors, predict_rating, i get_recommendations definides.


In [ ]:
# Cel·la 6: Execució d'un exemple de recomanació

# --- Paràmetres ---
TARGET_USER_ID = 1          # L'usuari per al qual volem recomanacions
K_NEIGHBORS = 6039          # Nombre de veïns a utilitzar (TODOS)
N_MOVIES_TO_RECOMMEND = 100  # Quantes pel·lícules volem recomanar

# --- Assegurar que les dades existeixen ---
if ('sim_matrix_pearson' in locals() and 
    'users_ratings_train' in locals() and 
    users_ratings_train is not None):
    
    print(f"--- Iniciant recomanació per a l'Usuari: {TARGET_USER_ID} ---")
    
    # Seleccionem la matriu de similitud de Pearson (generalment millor per a ratings)
    # Es podria canviar per sim_matrix_euclidean si es desitja provar
    active_similarity_matrix = sim_matrix_pearson
    
    start_time_rec = time.time()
    
    # Cridem a la funció principal
    recommendations = get_recommendations(
        target_user_id=TARGET_USER_ID,
        similarity_matrix=active_similarity_matrix,
        users_ratings_train=users_ratings_train,
        k=K_NEIGHBORS,
        n_recommendations=N_MOVIES_TO_RECOMMEND
    )
    
    end_time_rec = time.time()
    
    print(f"Càlcul de recomanacions finalitzat en {end_time_rec - start_time_rec:.4f} segons.")
    print("\n--- Top Recomanacions (Puntuació Predita, MovieID) ---")
    
    if recommendations:
        for pred_rating, movie_id in recommendations:
            print(f"Predicció: {pred_rating:.4f} \t| MovieID: {movie_id}")
    else:
        print("No s'han pogut generar recomanacions per a aquest usuari amb els paràmetres actuals.")

else:
    print("ERROR: Les variables 'sim_matrix_pearson' o 'users_ratings_train' no estan definides.")
    print("Si us plau, executi les cel·les anteriors (Pas 1 i 2) primer.")

--- Iniciant recomanació per a l'Usuari: 1 ---
Càlcul de recomanacions finalitzat en 0.1120 segons.

--- Top Recomanacions (Puntuació Predita, MovieID) ---
Predicció: 5.0000 	| MovieID: 3929
Predicció: 5.0000 	| MovieID: 3922
Predicció: 5.0000 	| MovieID: 3867
Predicció: 5.0000 	| MovieID: 3853
Predicció: 5.0000 	| MovieID: 3851
Predicció: 5.0000 	| MovieID: 3823
Predicció: 5.0000 	| MovieID: 3739
Predicció: 5.0000 	| MovieID: 3719
Predicció: 5.0000 	| MovieID: 3637
Predicció: 5.0000 	| MovieID: 3569
Predicció: 5.0000 	| MovieID: 3470
Predicció: 5.0000 	| MovieID: 3447
Predicció: 5.0000 	| MovieID: 3379
Predicció: 5.0000 	| MovieID: 3364
Predicció: 5.0000 	| MovieID: 3292
Predicció: 5.0000 	| MovieID: 3284
Predicció: 5.0000 	| MovieID: 3224
Predicció: 5.0000 	| MovieID: 3138
Predicció: 5.0000 	| MovieID: 3112
Predicció: 5.0000 	| MovieID: 3092
Predicció: 5.0000 	| MovieID: 3077
Predicció: 5.0000 	| MovieID: 2994
Predicció: 5.0000 	| MovieID: 2970
Predicció: 5.0000 	| MovieID: 2940
Pred